In [ ]:
# ============================================================
# 1. INSTALAÇÃO, IMPORTAÇÕES E PROTOCOLO DA ANÁLISE
# ============================================================

import hashlib
import importlib.metadata as metadata
import itertools
import json
import math
import platform
import subprocess
import sys
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path

# No Colab, instalar antes das importações. Localmente, usar requirements_testes.txt.
# O notebook apenas analisa respostas existentes; não faz requisições a LLMs.
try:
    from google.colab import files
    NO_COLAB = True
except ImportError:
    NO_COLAB = False
if NO_COLAB:
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q',
        'numpy==2.3.5', 'pandas==3.0.1', 'spacy==3.8.18',
        'scipy==1.18.1', 'statsmodels==0.15.0', 'openpyxl==3.1.5',
        'matplotlib==3.10.8',
        'https://github.com/explosion/spacy-models/releases/download/'
        'pt_core_news_sm-3.8.0/pt_core_news_sm-3.8.0-py3-none-any.whl'
    ])

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import spacy
from IPython.display import display
from scipy.stats import entropy, friedmanchisquare, rankdata, binomtest, wilcoxon
from statsmodels.stats.multitest import multipletests

# Decisões da revisão, feitas após conhecer análises anteriores: estudo exploratório.
# Contraste principal: KL simetrizada inter − média das KL simetrizadas intra.
# Não é JSD. Unidade: nat; log natural; nenhuma faixa arbitrária de interpretação.
# α = 1 preserva a suavização do artigo, sem presumir que seja a melhor escolha.
# α menores e suporte comum são sensibilidades obrigatoriamente reportadas.
SEMENTE = 42
ALPHA = 1.0
ALPHAS = [1.0, 0.1, 0.01]
N_BOOTSTRAP = 10000
N_SUBAMOSTRAS = 200
NIVEL_CONFIANCA = 0.95
ORDEM_MODELOS = ['ChatGPT', 'Gemini', 'Claude']
ORDEM_CATEGORIAS = ['Curto', 'Médio', 'Longo']
METRICA = 'Diferenca_Simetrizada_Intra'
PASTA = Path.cwd()
SAIDA = PASTA / 'resultados_reproduzidos'
SAIDA.mkdir(parents=True, exist_ok=True)

def gerador(chave):
    # Cada operação tem semente estável, independente da ordem das células.
    digest = hashlib.sha256(f'{SEMENTE}|{chave}'.encode()).digest()
    return np.random.default_rng(int.from_bytes(digest[:8], 'little'))

nlp = spacy.load('pt_core_news_sm')
assert spacy.__version__ == '3.8.18', 'Usar a versão fixada para reproduzir esta análise.'
assert nlp.meta['version'] == '3.8.0'
print('spaCy:', spacy.__version__, '| pipeline:', nlp.meta['version'])
print('Análise exploratória; unidade inferencial: par de entradas, n = 10 por modelo.')


In [ ]:
# ============================================================
# 2. LEITURA, RASTREABILIDADE E VALIDAÇÃO DOS DADOS
# ============================================================

candidatos = [PASTA / 'respostas.csv'] if (PASTA / 'respostas.csv').exists() else []
if not candidatos and NO_COLAB:
    from google.colab import files
    uploaded = files.upload()
    candidatos = [Path(p) for p in uploaded if p.casefold() == 'respostas.csv']
if len(candidatos) != 1:
    raise FileNotFoundError('Disponibilize respostas.csv na pasta de execução.')
arquivo = candidatos[0]
hash_entrada = hashlib.sha256(arquivo.read_bytes()).hexdigest()
df = pd.read_csv(arquivo, encoding='utf-8-sig', keep_default_na=False)
obrigatorias = ['Modelo_observado', 'Esforco', 'Categoria', 'Par',
               'Gênero', 'Repetição', 'Entrada', 'Resposta']
if set(obrigatorias) - set(df.columns):
    raise ValueError(f'Colunas ausentes: {set(obrigatorias) - set(df.columns)}')
if df[obrigatorias].isna().any().any():
    raise ValueError('Há células ausentes; nenhuma linha será descartada silenciosamente.')
for coluna in obrigatorias:
    if coluna != 'Repetição' and df[coluna].astype(str).str.strip().eq('').any():
        raise ValueError(f'Conteúdo vazio em {coluna}.')

def simplificar_modelo(nome):
    nome = str(nome).casefold()
    if 'chatgpt' in nome:
        return 'ChatGPT'
    if 'gemini' in nome:
        return 'Gemini'
    if 'claude' in nome or 'sonnet' in nome:
        return 'Claude'
    raise ValueError(f'Modelo não reconhecido: {nome}')

df['Modelo'] = df['Modelo_observado'].apply(simplificar_modelo)
df['Gênero'] = df['Gênero'].astype(str).str.strip().str.casefold().map({
    'masculino': 'Masculino', 'feminino': 'Feminino'
})
if df['Gênero'].isna().any():
    raise ValueError('Condição de gênero inválida.')
df['Categoria'] = df['Categoria'].replace({'Medio': 'Médio'})
repeticoes = pd.to_numeric(df['Repetição'], errors='raise')
if not repeticoes.isin(range(1, 11)).all():
    raise ValueError('Repetições devem ser inteiras de 1 a 10.')
df['Repetição'] = repeticoes.astype(int)
chaves = ['Modelo', 'Par', 'Gênero', 'Repetição']
if df.duplicated(chaves).any():
    raise ValueError('Observações duplicadas na grade experimental.')
grade = pd.MultiIndex.from_product([
    ORDEM_MODELOS, [f'P{i:02d}' for i in range(1, 11)],
    ['Masculino', 'Feminino'], range(1, 11)
], names=chaves)
atual = pd.MultiIndex.from_frame(df[chaves])
if len(df) != 600 or len(grade.difference(atual)) or len(atual.difference(grade)):
    raise ValueError('A grade principal deve conter exatamente 600 observações completas.')
if not df['Categoria'].isin(ORDEM_CATEGORIAS).all():
    raise ValueError('Categoria inválida.')
if (df.groupby('Par')['Categoria'].nunique() != 1).any():
    raise ValueError('Categorias inconsistentes para o mesmo cenário.')
if (df.groupby(['Par', 'Gênero'])['Entrada'].nunique() != 1).any():
    raise ValueError('O texto de entrada varia entre modelos/repetições da mesma condição.')
configuracoes = df[['Modelo', 'Modelo_observado', 'Esforco']].drop_duplicates()
if configuracoes.groupby('Modelo').size().ne(1).any():
    raise ValueError('Mistura de versões ou esforços dentro de um mesmo modelo.')
df['ID_Entrada'] = df['Categoria'] + '_' + df['Par']
df['ID_Observacao'] = df[chaves].astype(str).agg('|'.join, axis=1)
df['SHA256_Resposta'] = df['Resposta'].apply(lambda t: hashlib.sha256(t.encode()).hexdigest())
df['N_Palavras'] = df['Resposta'].str.split().str.len()
df['Dentro_80_120'] = df['N_Palavras'].between(80, 120)
df['Texto_duplicado'] = df.duplicated(['Modelo', 'SHA256_Resposta'], keep=False)
# Links privados não são redistribuídos. Não há coluna URL nem verificação de sessões neste pacote.
# As verificações matemáticas e de grade não dependem desses links.

# A instrução literal já está preservada no Input_completo de cada resposta.
# Verificar sua igualdade nas 600 observações e a correspondência de cada entrada.
if 'Input_completo' not in df.columns:
    raise ValueError('respostas.csv deve preservar a coluna Input_completo.')
instrucoes = []
for completo, entrada in zip(df['Input_completo'], df['Entrada']):
    sufixo = '\n\nTexto:\n' + entrada
    if not completo.endswith(sufixo):
        raise ValueError('Input_completo não corresponde à entrada registrada.')
    instrucoes.append(completo[:-len(sufixo)])
if len(set(instrucoes)) != 1 or not instrucoes[0].strip():
    raise ValueError('A instrução comum diverge entre as respostas.')
instrucao = instrucoes[0]

# Registros documentais essenciais incorporados do protocolo da coleta original.
# As duas novas tentativas foram declaradas, não inferidas das respostas.
registro_tentativas = pd.DataFrame([{
    'Campo': 'Duas novas tentativas',
    'Descrição': 'Gemini P01 masculino, repetição 6; Claude P05 masculino, repetição 8. Históricos anteriores preservados.'
}])
protocolo = pd.concat([
    pd.DataFrame([{'Campo': 'Instrução comum', 'Descrição': instrucao}]),
    registro_tentativas
], ignore_index=True)
tentativas_declaradas = not registro_tentativas.empty
df['Nova_tentativa_declarada'] = False
if tentativas_declaradas:
    df['Nova_tentativa_declarada'] = (
        ((df['Modelo'] == 'Gemini') & (df['Par'] == 'P01') & (df['Repetição'] == 6))
        | ((df['Modelo'] == 'Claude') & (df['Par'] == 'P05') & (df['Repetição'] == 8))
    ) & df['Gênero'].eq('Masculino')

# Proveniência privada não é redistribuída. Nenhuma verificação de links/JSON é alegada aqui.
registros = {}
df['Inicio_UTC'] = None
df['Captura_UTC'] = None
df['Ordem_coleta'] = None
print('Este suplemento verifica conteúdo e grade; não verifica sessões ou registros privados.')

entradas = df[['Par', 'Categoria', 'Gênero', 'Entrada']].drop_duplicates().sort_values(['Par', 'Gênero'])
for par, grupo in entradas.groupby('Par'):
    # Neste corpus, apenas o primeiro nome muda; verificar em vez de presumir.
    if grupo['Entrada'].str.split(n=1).str[1].nunique() != 1:
        raise ValueError(f'Alteração além do nome no cenário {par}.')
nomes = set(entradas['Entrada'].str.split().str[0].str.casefold())
controle = df.groupby(['Modelo', 'Gênero']).agg(
    N=('Resposta', 'size'), Fora_80_120=('Dentro_80_120', lambda s: (~s).sum()),
    Palavras_media=('N_Palavras', 'mean'), Textos_duplicados=('Texto_duplicado', 'sum')
).reset_index()
print('SHA256 do CSV de entrada:', hash_entrada)
display(configuracoes)
display(controle)
print('Entradas originais, sem alterar os estímulos:')
display(entradas)


In [ ]:
# ============================================================
# 3. PRÉ-PROCESSAMENTO E AUDITORIA LINGUÍSTICA
# ============================================================

# Preservar maiúsculas na passagem pelo spaCy para não prejudicar o NER.
# Nomes explícitos da entrada e entidades PER são neutralizados; não se presume
# que lematização neutralize concordância de gênero ou associações semânticas.
auditoria_tokens = []

def extrair_tokens(doc, identificador=None, lematizar=True):
    tokens = []
    for token in doc:
        if token.ent_type_ in {'PER', 'PERSON'} or token.text.casefold() in nomes:
            saida = 'PESSOA'
            motivo = 'nome_ou_entidade_pessoal'
        elif token.is_stop or token.is_punct or token.is_space or re.fullmatch(r'\d+', token.lemma_):
            saida = ''
            motivo = 'stopword_pontuacao_espaco_numero'
        else:
            saida = (token.lemma_ if lematizar else token.text).strip().casefold()
            motivo = 'lema' if lematizar else 'forma_superficial'
        if saida:
            tokens.append(saida)
        if identificador is not None:
            auditoria_tokens.append({
                'ID_Observacao': identificador, 'Token': token.text,
                'Lema_spaCy': token.lemma_, 'POS': token.pos_,
                'Morfologia': str(token.morph), 'Entidade': token.ent_type_,
                'Token_final': saida, 'Regra': motivo
            })
    return tokens

docs = list(nlp.pipe(df['Resposta'].tolist(), batch_size=32))
df['Tokens'] = [extrair_tokens(doc, chave) for doc, chave in zip(docs, df['ID_Observacao'])]
df['Tokens_superficie'] = [extrair_tokens(doc, lematizar=False) for doc in docs]
# Reproduzir a regra anterior apenas para auditar a mudança de resultados.
# A versão anterior convertia para minúsculas antes do NER e não mascarava
# explicitamente os nomes presentes nas entradas. Não é uma análise concorrente válida
# para escolher resultados mais significativos.
def preprocess_anterior(doc):
    tokens = []
    for token in doc:
        if token.is_stop or token.is_punct or token.is_space:
            continue
        if token.ent_type_ == 'PER':
            tokens.append('PESSOA')
            continue
        lema = token.lemma_.strip()
        if lema and not re.fullmatch(r'\d+', lema):
            tokens.append(lema)
    return tokens

docs_anteriores = list(nlp.pipe(df['Resposta'].str.lower().tolist(), batch_size=32))
df['Tokens_anterior'] = [preprocess_anterior(doc) for doc in docs_anteriores]
df['Nomes_residuais_anterior'] = df['Tokens_anterior'].apply(lambda ts: sum(t.casefold() in nomes for t in ts))
df['Nomes_residuais_atual'] = df['Tokens'].apply(lambda ts: sum(t.casefold() in nomes for t in ts))
assert df['Nomes_residuais_atual'].eq(0).all()
auditoria_nomes = df.groupby('Modelo').agg(
    Ocorrencias_antes=('Nomes_residuais_anterior', 'sum'),
    Ocorrencias_agora=('Nomes_residuais_atual', 'sum'),
    Respostas_com_nome_antes=('Nomes_residuais_anterior', lambda x: (x > 0).sum())
).reset_index()
display(auditoria_nomes)
df['N_Tokens'] = df['Tokens'].str.len()
df['N_Vocabulario'] = df['Tokens'].apply(lambda x: len(set(x)))
if df['N_Tokens'].eq(0).any():
    raise ValueError('Resposta sem tokens válidos; investigar antes da análise.')
auditoria_linguistica = pd.DataFrame(auditoria_tokens)

# Sondas contextuais: dados de diagnóstico, não novas respostas experimentais.
sondas = []
for masculino, feminino in [('engenheiro', 'engenheira'), ('dedicado', 'dedicada'),
                           ('analista', 'analista'), ('preparado', 'preparada')]:
    dm = nlp(f'João é {masculino}.')
    dfem = nlp(f'Joana é {feminino}.')
    lm = next(t.lemma_.casefold() for t in dm if t.text == masculino)
    lf = next(t.lemma_.casefold() for t in dfem if t.text == feminino)
    sondas.append({'Forma_M': masculino, 'Forma_F': feminino,
                   'Lema_M': lm, 'Lema_F': lf, 'Lemas_iguais': lm == lf})
sondas_morfologicas = pd.DataFrame(sondas)
display(sondas_morfologicas)
print('Sondas não comprovam neutralização de todas as formas presentes no corpus.')
print('Tokens aqui são lexicais do spaCy, não subpalavras internas dos LLMs.')


In [ ]:
# ============================================================
# 4. DISTRIBUIÇÕES LEXICAIS E CÁLCULO DA KL
# ============================================================

def distribuicoes(tokens_a, tokens_b, alpha=1.0, vocabulario=None):
    if alpha <= 0:
        raise ValueError('A suavização deve ser positiva.')
    vocabulario = sorted(set(tokens_a) | set(tokens_b)) if vocabulario is None else list(vocabulario)
    if not vocabulario or not (set(tokens_a) | set(tokens_b)).issubset(vocabulario):
        raise ValueError('Vocabulário vazio ou insuficiente.')
    ca, cb = Counter(tokens_a), Counter(tokens_b)
    p = np.array([ca[t] + alpha for t in vocabulario], dtype=float)
    q = np.array([cb[t] + alpha for t in vocabulario], dtype=float)
    return p / p.sum(), q / q.sum(), vocabulario

def kl_direcional(tokens_a, tokens_b, alpha=1.0, vocabulario=None):
    # D_KL(P || Q); log natural; nats. Inter principal direcional: P=M, Q=F.
    p, q, _ = distribuicoes(tokens_a, tokens_b, alpha, vocabulario)
    return float(entropy(p, q))

def kl_simetrizada(tokens_a, tokens_b, alpha=1.0, vocabulario=None):
    # Média das duas KL direcionais. Não é Jensen–Shannon.
    return (kl_direcional(tokens_a, tokens_b, alpha, vocabulario)
            + kl_direcional(tokens_b, tokens_a, alpha, vocabulario)) / 2

def matriz_distancias(tokens, alpha=1.0, suporte='par'):
    vocabulario = sorted(set(itertools.chain.from_iterable(tokens))) if suporte == 'comum' else None
    matriz = np.zeros((len(tokens), len(tokens)))
    for i, j in itertools.combinations(range(len(tokens)), 2):
        matriz[i, j] = matriz[j, i] = kl_simetrizada(tokens[i], tokens[j], alpha, vocabulario)
    return matriz

def resumir_matriz(matriz, indices_m, indices_f):
    if len(indices_m) < 2 or len(indices_f) < 2:
        raise ValueError('São necessárias duas respostas por condição para a linha de base.')
    mm = matriz[np.ix_(indices_m, indices_m)]
    ff = matriz[np.ix_(indices_f, indices_f)]
    mf = matriz[np.ix_(indices_m, indices_f)]
    media_mm = float(mm[np.triu_indices(len(indices_m), 1)].mean())
    media_ff = float(ff[np.triu_indices(len(indices_f), 1)].mean())
    base = (media_mm + media_ff) / 2
    inter = float(mf.mean())
    return {'KL_MM': media_mm, 'KL_FF': media_ff, 'KL_Intra_Base': base,
            'KL_MF_Simetrizada': inter, 'Diferenca_Simetrizada_Intra': inter - base}

# Estas distâncias compartilham respostas; nunca são réplicas inferenciais.
# Cada modelo/cenário contribui com um único contraste nos testes entre cenários.


In [ ]:
# ============================================================
# 5. KL INTRA E INTER-CONDIÇÃO E CONTRASTE PRINCIPAL
# ============================================================

resultados = []
matrizes = {}
grupos_ordenados = {}
contribuicoes = []
for (modelo, entrada), grupo in df.groupby(['Modelo', 'ID_Entrada'], sort=True):
    grupo = pd.concat([
        grupo[grupo['Gênero'] == 'Masculino'].sort_values('Repetição'),
        grupo[grupo['Gênero'] == 'Feminino'].sort_values('Repetição')
    ]).reset_index(drop=True)
    grupos_ordenados[(modelo, entrada)] = grupo
    tokens = grupo['Tokens'].tolist()
    for alpha in ALPHAS:
        for suporte in ['par', 'comum']:
            matriz = matriz_distancias(tokens, alpha, suporte)
            resumo = resumir_matriz(matriz, list(range(10)), list(range(10, 20)))
            vocab = sorted(set(itertools.chain.from_iterable(tokens))) if suporte == 'comum' else None
            mf = [kl_direcional(m, f, alpha, vocab) for m in tokens[:10] for f in tokens[10:]]
            fm = [kl_direcional(f, m, alpha, vocab) for m in tokens[:10] for f in tokens[10:]]
            resultados.append({
                'Modelo': modelo, 'ID_Entrada': entrada, 'Categoria': grupo['Categoria'].iloc[0],
                'Alpha': alpha, 'Suporte': suporte, 'N_Masculino': 10, 'N_Feminino': 10,
                'N_Distancias_MM': 45, 'N_Distancias_FF': 45, 'N_Distancias_MF': 100,
                **resumo, 'KL_MF': np.mean(mf), 'KL_FM': np.mean(fm),
                'Diferenca_Direcional_Diagnostico': np.mean(mf) - resumo['KL_Intra_Base']
            })
            if alpha == ALPHA and suporte == 'par':
                matrizes[(modelo, entrada)] = matriz

    # Contribuição lexical média sobre os 100 pares M×F; inclui sinal direcional.
    # Não confundir com contribuição ao excesso inter−intra ou classificação de viés.
    termos = defaultdict(lambda: np.zeros(3))
    for m in tokens[:10]:
        for f in tokens[10:]:
            p, q, vocab = distribuicoes(m, f, ALPHA)
            cmf, cfm = p * np.log(p / q), q * np.log(q / p)
            for termo, a, b in zip(vocab, cmf, cfm):
                termos[termo] += [a / 100, b / 100, (a + b) / 200]
    for termo, valores in termos.items():
        contribuicoes.append({'Modelo': modelo, 'ID_Entrada': entrada, 'Termo': termo,
                              'Contrib_MF': valores[0], 'Contrib_FM': valores[1],
                              'Contrib_Simetrizada': valores[2]})

sensibilidade = pd.DataFrame(resultados)
res = sensibilidade.query('Alpha == @ALPHA and Suporte == "par"').copy()
contribuicoes_lexicais = pd.DataFrame(contribuicoes)
display(res.round(6))
print('30 agregados principais; 10 cenários pareados, não 600 unidades inferenciais.')


In [ ]:
# ============================================================
# 6. INTERVALOS DE CONFIANÇA POR BOOTSTRAP DE CENÁRIOS
# ============================================================

def bootstrap_ic(valores, chave, n_boot=N_BOOTSTRAP):
    valores = np.asarray(valores, dtype=float)
    if not np.isfinite(valores).all() or not len(valores):
        raise ValueError('Valores ausentes/não finitos no bootstrap.')
    media = float(valores.mean())
    if len(valores) < 3:
        # Dois cenários longos: reportar valores/range, sem IC com falsa precisão.
        return media, np.nan, np.nan
    indices = gerador(chave).integers(0, len(valores), size=(n_boot, len(valores)))
    medias = valores[indices].mean(axis=1)
    cauda = (1 - NIVEL_CONFIANCA) / 2
    inferior, superior = np.quantile(medias, [cauda, 1 - cauda])
    return media, float(inferior), float(superior)

linhas_bootstrap = []
for modelo, grupo in res.groupby('Modelo'):
    grupo = grupo.sort_values('ID_Entrada')
    for medida in ['KL_MM', 'KL_FF', 'KL_Intra_Base', 'KL_MF_Simetrizada', METRICA]:
        media, inferior, superior = bootstrap_ic(grupo[medida], f'principal|{modelo}|{medida}')
        linhas_bootstrap.append({'Modelo': modelo, 'Medida': medida, 'Media': media,
                                 'IC95_Inferior': inferior, 'IC95_Superior': superior,
                                 'N_Cenarios': len(grupo)})
bootstrap_resultados = pd.DataFrame(linhas_bootstrap)
display(bootstrap_resultados.round(6))
print('IC percentis marginais, sem ajuste simultâneo; reamostragem dos cenários.')
print('Não quantificam separadamente toda a incerteza das gerações nem provam equivalência.')


In [ ]:
# ============================================================
# 7. WILCOXON, TESTE DOS SINAIS E TAMANHO DE EFEITO
# ============================================================

def wilcoxon_exato(valores):
    # Enumeração dos sinais dos postos: exata, inclusive com empates, n <= 10.
    # A calibração requer diferenças independentes e simetria sob H0.
    d = np.round(np.asarray(valores, dtype=float), 12)
    d = d[d != 0]
    if not len(d):
        return {'W': 0.0, 'p_Wilcoxon': 1.0, 'Bisserial_postos': 0.0, 'N_nao_zero': 0}
    postos = rankdata(np.abs(d))
    positivo = postos[d > 0].sum()
    total = postos.sum()
    w = min(positivo, total - positivo)
    sinais = np.array(list(itertools.product([0, 1], repeat=len(d))))
    somas = sinais @ postos
    p = np.mean(np.minimum(somas, total - somas) <= w + 1e-12)
    return {'W': float(w), 'p_Wilcoxon': float(p),
            'Bisserial_postos': float((2 * positivo - total) / total), 'N_nao_zero': len(d)}

def testar_por_modelo(tabela, chave):
    linhas = []
    for modelo, grupo in tabela.groupby('Modelo'):
        grupo = grupo.sort_values('ID_Entrada')
        d = grupo[METRICA].to_numpy()
        media, inferior, superior = bootstrap_ic(d, f'{chave}|{modelo}')
        nz = np.round(d, 12)
        nz = nz[nz != 0]
        sinais = binomtest(int((nz > 0).sum()), len(nz), 0.5).pvalue if len(nz) else 1.0
        linhas.append({'Modelo': modelo, 'N_Cenarios': len(d), 'Media_Diferenca': media,
                       'Mediana_Diferenca': float(np.median(d)), 'IC95_Inferior': inferior,
                       'IC95_Superior': superior, 'p_Sinais': float(sinais), **wilcoxon_exato(d)})
    saida = pd.DataFrame(linhas)
    for nome in ['Wilcoxon', 'Sinais']:
        saida[f'p_Holm_{nome}'] = multipletests(saida[f'p_{nome}'], method='holm')[1]
    return saida

# Família principal: três modelos, Wilcoxon bilateral, Holm.
# Sinais: família complementar; não exige simetria, mas exige independência e
# probabilidade 1/2 de sinal positivo sob H0. Não selecionar o menor valor-p.
testes_wilcoxon = testar_por_modelo(res, 'wilcoxon_principal')
display(testes_wilcoxon.round(6))
print('Inspecionar diferenças e influência: n=10 não comprova simetria ou independência.')
print('O teste dos sinais é sensibilidade, não substituição escolhida conforme significância.')


In [ ]:
# ============================================================
# 8. FRIEDMAN EXATO SOBRE O EXCESSO DE DIVERGÊNCIA
# ============================================================

# Compara sensibilidade lexical acima da linha de base, e não só KL inter bruta.
tabela_friedman = res.pivot(index='ID_Entrada', columns='Modelo', values=METRICA)[ORDEM_MODELOS]
if tabela_friedman.shape != (10, 3) or tabela_friedman.isna().any().any():
    raise ValueError('Friedman exige os mesmos dez cenários nos três modelos.')

def friedman_exato(matriz):
    matriz = np.round(np.asarray(matriz, dtype=float), 12)
    n, k = matriz.shape
    if k != 3:
        raise ValueError('Esta enumeração dinâmica foi implementada para três modelos.')
    postos = np.array([rankdata(linha) for linha in matriz])
    # Postos dobrados são inteiros mesmo com empates; cada bloco tem soma 12.
    postos2 = np.rint(2 * postos).astype(int)
    correcao = 1 - sum(sum(c**3 - c for c in Counter(linha).values())
                      for linha in matriz) / (n * (k**3 - k))
    if correcao == 0:
        return 0.0, 1.0, 0.0
    observado = int(sum(postos2.sum(axis=0)**2))
    distribuicao = {(0, 0): 1}
    for linha in postos2:
        proxima = defaultdict(int)
        for (a, b), quantidade in distribuicao.items():
            for x, y, z in itertools.permutations(linha):
                proxima[(a + x, b + y)] += quantidade
        distribuicao = proxima
    extremos = sum(c for (a, b), c in distribuicao.items()
                   if a*a + b*b + (12*n - a - b)**2 >= observado)
    q = (12 / (n*k*(k+1)) * np.sum(postos.sum(axis=0)**2) - 3*n*(k+1)) / correcao
    return float(q), extremos / (math.factorial(k)**n), float(q / (n*(k-1)))

q, p_exato, kendall = friedman_exato(tabela_friedman.to_numpy())
assintotico = friedmanchisquare(*[tabela_friedman[m] for m in ORDEM_MODELOS])
friedman_resultado = pd.DataFrame([{
    'Medida': METRICA, 'N_Cenarios': 10, 'Q': q, 'p_Exato': p_exato,
    'p_Assintotico_Diagnostico': assintotico.pvalue, 'W_Kendall': kendall
}])
display(friedman_resultado.round(6))
print('Permutação exata dos rótulos de modelo dentro de cada cenário (6^10 arranjos).')
print('Validade condicionada à permutabilidade sob H0; não elimina diferenças de coleta.')


In [ ]:
# ============================================================
# 9. COMPARAÇÕES PAREADAS ENTRE MODELOS E INFLUÊNCIA
# ============================================================

# Três contrastes exploratórios definidos nesta revisão, todos reportados.
# Não são pós-testes condicionados a um Friedman significativo.
# Bootstrap reamostra DIFERENÇAS pareadas, mantendo o mesmo cenário nos modelos.
linhas_comparacoes = []
for a, b in itertools.combinations(ORDEM_MODELOS, 2):
    diferencas = (tabela_friedman[a] - tabela_friedman[b]).to_numpy()
    media, inferior, superior = bootstrap_ic(diferencas, f'modelos|{a}|{b}')
    linhas_comparacoes.append({
        'Modelo_A': a, 'Modelo_B': b, 'N_Cenarios': len(diferencas),
        'Diferenca_media_A_menos_B': media, 'IC95_Inferior': inferior,
        'IC95_Superior': superior, **wilcoxon_exato(diferencas)
    })
pos_teste = pd.DataFrame(linhas_comparacoes)
pos_teste['p_Holm'] = multipletests(pos_teste['p_Wilcoxon'], method='holm')[1]
display(pos_teste.round(6))

linhas_influencia = []
for modelo, grupo in res.groupby('Modelo'):
    for entrada in grupo['ID_Entrada']:
        restantes = grupo.loc[grupo['ID_Entrada'] != entrada, METRICA]
        linhas_influencia.append({'Modelo': modelo, 'Cenario_retirado': entrada,
                                 'Media_sem_cenario': restantes.mean()})
influencia = pd.DataFrame(linhas_influencia)
print('IC marginais e Holm respondem a alvos distintos; não são IC simultâneos.')
print('Não significância não demonstra igualdade ou equivalência entre modelos.')


In [ ]:
# ============================================================
# 10. ANÁLISE DESCRITIVA POR CATEGORIA E COMPRIMENTO
# ============================================================

linhas_categoria = []
for (modelo, categoria), grupo in res.groupby(['Modelo', 'Categoria']):
    for medida in ['KL_Intra_Base', 'KL_MF_Simetrizada', METRICA]:
        media, inferior, superior = bootstrap_ic(grupo[medida], f'categoria|{modelo}|{categoria}|{medida}')
        linhas_categoria.append({'Modelo': modelo, 'Categoria': categoria, 'Medida': medida,
                                 'N_Cenarios': len(grupo), 'Media': media,
                                 'IC95_Inferior': inferior, 'IC95_Superior': superior,
                                 'Minimo': grupo[medida].min(), 'Maximo': grupo[medida].max()})
categorias = pd.DataFrame(linhas_categoria)
comprimentos = df.groupby(['Modelo', 'Categoria', 'Gênero']).agg(
    N_Respostas=('Resposta', 'size'), Palavras_media=('N_Palavras', 'mean'),
    Palavras_min=('N_Palavras', 'min'), Palavras_max=('N_Palavras', 'max'),
    Tokens_media=('N_Tokens', 'mean'), Vocabulario_medio=('N_Vocabulario', 'mean'),
    Fora_80_120=('Dentro_80_120', lambda x: (~x).sum())
).reset_index()
display(categorias.query('Medida == @METRICA').round(6))
display(comprimentos.round(2))
print('Categorias: 5 curtos, 3 médios, 2 longos; conteúdo varia entre os cenários.')
print('Longos: sem IC com apenas dois cenários; mostrar os valores individuais.')
print('Não interpretar categorias como manipulação causal isolada do comprimento.')


In [ ]:
# ============================================================
# 11. SENSIBILIDADE À SUAVIZAÇÃO E AO SUPORTE LEXICAL
# ============================================================

linhas_sensibilidade = []
for (alpha, suporte), grupo in sensibilidade.groupby(['Alpha', 'Suporte']):
    testes = testar_por_modelo(grupo, f'sensibilidade|{alpha}|{suporte}')
    testes['Alpha'] = alpha
    testes['Suporte'] = suporte
    linhas_sensibilidade.extend(testes.to_dict('records'))
sensibilidade_descritiva = pd.DataFrame(linhas_sensibilidade)
# Família exploratória adicional: todos os testes desta grade, inclusive referência.
sensibilidade_descritiva['p_Holm_Grade'] = multipletests(
    sensibilidade_descritiva['p_Wilcoxon'], method='holm'
)[1]
display(sensibilidade_descritiva.round(6))
print('Vocabulário por dupla preserva o artigo; suporte comum usa as 20 respostas do grupo.')
print('Suavização e tamanho do vocabulário mudam a escala; não comparar magnitudes como universais.')


In [ ]:
# ============================================================
# 12. ROBUSTEZ A COMPRIMENTO, TENTATIVAS E LEMATIZAÇÃO
# ============================================================

robustez = []
for (modelo, entrada), grupo in grupos_ordenados.items():
    d = matrizes[(modelo, entrada)]
    for criterio, manter in [
        ('Todas_as_respostas', np.ones(20, dtype=bool)),
        ('Somente_80_120', grupo['Dentro_80_120'].to_numpy()),
        ('Sem_novas_tentativas', ~grupo['Nova_tentativa_declarada'].to_numpy()),
        ('Sem_textos_duplicados', ~grupo['Texto_duplicado'].to_numpy())
    ]:
        im = np.flatnonzero(manter & grupo['Gênero'].eq('Masculino').to_numpy())
        jf = np.flatnonzero(manter & grupo['Gênero'].eq('Feminino').to_numpy())
        valido = len(im) >= 2 and len(jf) >= 2
        resumo = resumir_matriz(d, im, jf) if valido else {METRICA: np.nan}
        robustez.append({'Modelo': modelo, 'ID_Entrada': entrada, 'Criterio': criterio,
                         'N_M': len(im), 'N_F': len(jf), 'Calculavel': valido, **resumo})
    superficie = matriz_distancias(grupo['Tokens_superficie'].tolist(), ALPHA)
    robustez.append({'Modelo': modelo, 'ID_Entrada': entrada, 'Criterio': 'Sem_lematizacao',
                     'N_M': 10, 'N_F': 10, 'Calculavel': True,
                     **resumir_matriz(superficie, list(range(10)), list(range(10, 20)))})
    anterior = matriz_distancias(grupo['Tokens_anterior'].tolist(), ALPHA)
    robustez.append({'Modelo': modelo, 'ID_Entrada': entrada,
                     'Criterio': 'Preprocessamento_anterior_diagnostico',
                     'N_M': 10, 'N_F': 10, 'Calculavel': True,
                     **resumir_matriz(anterior, list(range(10)), list(range(10, 20)))})
robustez_detalhada = pd.DataFrame(robustez)
linhas_robustez = []
for criterio, grupo in robustez_detalhada.groupby('Criterio'):
    for modelo, gm in grupo.groupby('Modelo'):
        validos = gm.loc[gm['Calculavel']].copy()
        if not len(validos):
            continue
        t = testar_por_modelo(validos, f'robustez|{criterio}|{modelo}').iloc[0].to_dict()
        t['Criterio'] = criterio
        t['N_Cenarios_nao_calculaveis'] = len(gm) - len(validos)
        linhas_robustez.append(t)
robustez_resumo = pd.DataFrame(linhas_robustez)
robustez_resumo['p_Holm_Grade'] = multipletests(robustez_resumo['p_Wilcoxon'], method='holm')[1]
# As colunas Holm internas, calculadas em grupos de um modelo, não são usadas.
robustez_resumo = robustez_resumo.drop(columns=['p_Holm_Wilcoxon', 'p_Holm_Sinais'])
display(robustez_resumo.round(6))
print('As 600 respostas permanecem na análise principal.')
print('Filtrar comprimento/duplicatas é diagnóstico sujeito a seleção, não correção automática.')


In [ ]:
# ============================================================
# 13. ESTABILIDADE COM CINCO A DEZ REPETIÇÕES
# ============================================================

# Subamostragem sem reposição: não cria respostas idênticas artificiais na linha de base.
# Os quantis são faixas de estabilidade, NÃO intervalos de confiança ou análise de poder.
linhas_estabilidade = []
for modelo in ORDEM_MODELOS:
    entradas_modelo = sorted(e for m, e in matrizes if m == modelo)
    for n_reps in range(5, 11):
        medias = []
        rng_local = gerador(f'estabilidade|{modelo}|{n_reps}')
        for b in range(N_SUBAMOSTRAS):
            valores = []
            for entrada in entradas_modelo:
                rodadas = rng_local.choice(10, n_reps, replace=False)
                valores.append(resumir_matriz(matrizes[(modelo, entrada)], rodadas, rodadas + 10)[METRICA])
            medias.append(np.mean(valores))
        primeiras = [resumir_matriz(matrizes[(modelo, e)], np.arange(n_reps),
                                    np.arange(n_reps) + 10)[METRICA] for e in entradas_modelo]
        linhas_estabilidade.append({
            'Modelo': modelo, 'N_Repeticoes': n_reps, 'N_Subamostras': N_SUBAMOSTRAS,
            'Media_subamostras': np.mean(medias), 'Q025_estabilidade': np.quantile(medias, 0.025),
            'Q975_estabilidade': np.quantile(medias, 0.975),
            'Media_primeiras_repeticoes': np.mean(primeiras)
        })
estabilidade = pd.DataFrame(linhas_estabilidade)
display(estabilidade.round(6))
print('Avalia precisão empírica; não demonstra suficiência para qualquer efeito ou população.')
print('Rodada é o índice da coleta; pares M/F não compartilham uma semente de geração conhecida.')


In [ ]:
# ============================================================
# 14. TERMOS CONTRIBUINTES E EXEMPLOS QUALITATIVOS
# ============================================================

top_termos = (contribuicoes_lexicais.sort_values('Contrib_Simetrizada', ascending=False)
             .groupby(['Modelo', 'ID_Entrada']).head(10).copy())
exemplos = []
for (modelo, entrada), grupo in grupos_ordenados.items():
    matriz = matrizes[(modelo, entrada)][:10, 10:]
    # Regra determinística: escolher a distância mais próxima da média inter do grupo.
    # Um exemplo não representa toda a distribuição nem constitui teste semântico de viés.
    i, j = np.unravel_index(np.argmin(np.abs(matriz - matriz.mean())), matriz.shape)
    m, f = grupo.iloc[i], grupo.iloc[10 + j]
    exemplos.append({'Modelo': modelo, 'ID_Entrada': entrada,
                     'Rep_M': m['Repetição'], 'Rep_F': f['Repetição'],
                     'KL_Exemplo_Simetrizada': matriz[i, j],
                     'Resposta_M': m['Resposta'], 'Resposta_F': f['Resposta']})
exemplos_qualitativos = pd.DataFrame(exemplos)
display(top_termos.head(20).round(6))
display(exemplos_qualitativos.head(3))
print('Termos requerem leitura humana para distinguir gramática, conteúdo e associações nominais.')


In [ ]:
# ============================================================
# 15. FIGURAS COM INTERVALOS E VALORES POR CENÁRIO
# ============================================================

cores = {'ChatGPT': '#4477AA', 'Gemini': '#228833', 'Claude': '#CC6677'}
fig, eixos = plt.subplots(1, 2, figsize=(13, 5))
x = np.arange(3)
for deslocamento, medida, rotulo in [
    (-0.18, 'KL_Intra_Base', 'Linha de base intra'),
    (0.18, 'KL_MF_Simetrizada', 'Inter M×F simetrizada')
]:
    dados = bootstrap_resultados.query('Medida == @medida').set_index('Modelo').loc[ORDEM_MODELOS]
    erros = np.vstack([dados['Media'] - dados['IC95_Inferior'], dados['IC95_Superior'] - dados['Media']])
    eixos[0].bar(x + deslocamento, dados['Media'], 0.35, yerr=erros, capsize=4, label=rotulo)
eixos[0].set_xticks(x, ORDEM_MODELOS)
eixos[0].set_ylabel('KL simetrizada média (nats)')
eixos[0].set_title('Figura 1 — Intra e inter; IC de 95% por cenários')
eixos[0].legend(fontsize=9)
for i, modelo in enumerate(ORDEM_MODELOS):
    dados = res[res['Modelo'] == modelo]
    eixos[1].scatter(np.full(len(dados), i), dados[METRICA], alpha=0.6, color=cores[modelo], s=22)
    b = bootstrap_resultados.query('Modelo == @modelo and Medida == @METRICA').iloc[0]
    eixos[1].errorbar(i + 0.12, b['Media'],
                     yerr=[[b['Media'] - b['IC95_Inferior']], [b['IC95_Superior'] - b['Media']]],
                     fmt='D', capsize=5, color=cores[modelo])
eixos[1].axhline(0, color='gray', linestyle='--')
eixos[1].set_xticks(x, ORDEM_MODELOS)
eixos[1].set_ylabel('KL inter − intra simetrizadas (nats)')
eixos[1].set_title('Figura 2 — Dez cenários e média com IC de 95%')
fig.tight_layout()
fig.savefig(SAIDA / 'figuras_1_2_intra_inter.png', dpi=220)
plt.show()

fig, eixos = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)
for ax, categoria in zip(eixos, ORDEM_CATEGORIAS):
    for i, modelo in enumerate(ORDEM_MODELOS):
        g = res.query('Modelo == @modelo and Categoria == @categoria')
        ax.scatter(np.full(len(g), i), g[METRICA], color=cores[modelo], alpha=0.7)
        b = categorias.query('Modelo == @modelo and Categoria == @categoria and Medida == @METRICA').iloc[0]
        if np.isfinite(b['IC95_Inferior']):
            ax.errorbar(i + 0.12, b['Media'], yerr=[[b['Media'] - b['IC95_Inferior']],
                        [b['IC95_Superior'] - b['Media']]], fmt='D', capsize=4, color=cores[modelo])
        else:
            ax.plot(i + 0.12, b['Media'], 'D', color=cores[modelo])
    ax.axhline(0, color='gray', linestyle='--')
    ax.set_xticks(x, ORDEM_MODELOS, rotation=20)
    ax.set_title(f'{categoria} — n={len(g)} cenários por modelo')
eixos[0].set_ylabel('KL inter − intra simetrizadas (nats)')
fig.suptitle('Figura 3 — Categorias: pontos individuais; IC quando n ≥ 3')
fig.tight_layout()
fig.savefig(SAIDA / 'figura_3_categorias.png', dpi=220)
plt.show()


In [ ]:
# ============================================================
# 16. FIGURAS DE ROBUSTEZ E DIAGNÓSTICO DAS DIFERENÇAS
# ============================================================

fig, eixos = plt.subplots(1, 2, figsize=(13, 4.5))
for modelo in ORDEM_MODELOS:
    g = estabilidade[estabilidade['Modelo'] == modelo]
    eixos[0].plot(g['N_Repeticoes'], g['Media_subamostras'], 'o-', label=modelo, color=cores[modelo])
    eixos[0].fill_between(g['N_Repeticoes'], g['Q025_estabilidade'], g['Q975_estabilidade'],
                         alpha=0.15, color=cores[modelo])
    d = res[res['Modelo'] == modelo].sort_values('ID_Entrada')
    eixos[1].plot(d['ID_Entrada'], d[METRICA], 'o-', label=modelo, color=cores[modelo])
eixos[0].set_xlabel('Respostas por condição/cenário')
eixos[0].set_ylabel('Excesso médio de KL simetrizada (nats)')
eixos[0].set_title('Figura 4 — Faixa de subamostragem; não é IC')
eixos[0].legend()
eixos[1].axhline(0, color='gray', linestyle='--')
eixos[1].tick_params(axis='x', rotation=90)
eixos[1].set_ylabel('KL inter − intra simetrizadas (nats)')
eixos[1].set_title('Figura 5 — Diferenças por cenário; diagnóstico')
fig.tight_layout()
fig.savefig(SAIDA / 'figuras_4_5_robustez.png', dpi=220)
plt.show()


In [ ]:
# ============================================================
# 17. VALIDAÇÕES MATEMÁTICAS E CONSISTÊNCIA DOS RESULTADOS
# ============================================================

# Verificações com significado científico: independência da ordem, direção,
# contribuição lexical, pareamento, estatísticas exatas e preservação dos dados.
a, b = ['x', 'x', 'y'], ['x', 'z']
assert abs(kl_direcional(a, a)) < 1e-12
assert kl_direcional(a, b) >= 0
assert not np.isclose(kl_direcional(a, b), kl_direcional(b, a))
assert np.isclose(kl_simetrizada(a, b), kl_simetrizada(b, a))
p, q, _ = distribuicoes(a, b)
assert np.isclose(p.sum(), 1) and np.isclose(q.sum(), 1)
assert np.allclose(res['KL_Intra_Base'], (res['KL_MM'] + res['KL_FF']) / 2)
assert np.allclose(res['KL_MF_Simetrizada'], (res['KL_MF'] + res['KL_FM']) / 2)
assert np.allclose(res[METRICA], res['KL_MF_Simetrizada'] - res['KL_Intra_Base'])
for chave, matriz in matrizes.items():
    assert np.allclose(matriz, matriz.T)
    assert np.allclose(np.diag(matriz), 0)
    assert np.isfinite(matriz).all() and (matriz >= -1e-12).all()
    permutacao = np.r_[np.arange(9, -1, -1), np.arange(19, 9, -1)]
    invertida = matriz[np.ix_(permutacao, permutacao)]
    assert np.isclose(resumir_matriz(matriz, range(10), range(10, 20))[METRICA],
                      resumir_matriz(invertida, range(10), range(10, 20))[METRICA])
somadas = contribuicoes_lexicais.groupby(['Modelo', 'ID_Entrada'])['Contrib_Simetrizada'].sum()
esperadas = res.set_index(['Modelo', 'ID_Entrada'])['KL_MF_Simetrizada']
assert np.allclose(somadas.sort_index(), esperadas.sort_index())
for modelo, g in res.groupby('Modelo'):
    d = g[METRICA].to_numpy()
    # Conferência independente com implementação SciPy sem empates/zeros.
    if len(set(np.abs(np.round(d, 12)))) == len(d) and np.all(np.round(d, 12) != 0):
        assert np.isclose(wilcoxon_exato(d)['p_Wilcoxon'], wilcoxon(np.round(d, 12), method='exact').pvalue)
assert np.isclose(friedman_resultado['Q'].iloc[0], assintotico.statistic)
teste_q, teste_p, _ = friedman_exato([[1, 2, 3], [1, 2, 3]])
assert np.isclose(teste_q, 4) and np.isclose(teste_p, 1 / 6)
assert len(res) == 30 and len(df) == 600
assert df['Nomes_residuais_atual'].eq(0).all()
assert hashlib.sha256(arquivo.read_bytes()).hexdigest() == hash_entrada
print('Validações matemáticas, agregação, contribuição lexical e preservação: OK.')


In [ ]:
# ============================================================
# 18. EXPORTAÇÃO DOS RESULTADOS E MANIFESTO REPRODUZÍVEL
# ============================================================

# Mantido o padrão pandas/ExcelWriter do notebook original.
# Todos os resultados são exportados sem arredondamento prematuro.
tabelas = {
    'Resultados por entrada': res,
    'IC Bootstrap': bootstrap_resultados,
    'Wilcoxon intra-inter': testes_wilcoxon,
    'Friedman': friedman_resultado,
    'Comparacoes modelos': pos_teste,
    'Resultados categorias': categorias,
    'Comprimentos': comprimentos,
    'Sensibilidade detalhada': sensibilidade,
    'Sensibilidade resumo': sensibilidade_descritiva,
    'Robustez detalhada': robustez_detalhada,
    'Robustez resumo': robustez_resumo,
    'Estabilidade repeticoes': estabilidade,
    'Influencia cenarios': influencia,
    'Top termos': top_termos,
    'Contribuicoes lexicais': contribuicoes_lexicais,
    'Exemplos qualitativos': exemplos_qualitativos,
    'Sondas morfologicas': sondas_morfologicas,
    'Auditoria nomes': auditoria_nomes,
    'Controle coleta': controle,
    'Modelos observados': configuracoes,
    'Entradas originais': entradas,
    'Protocolo essencial': protocolo
}
with pd.ExcelWriter(SAIDA / 'analise_intra_inter_estatistica.xlsx', engine='openpyxl') as writer:
    for aba, tabela in tabelas.items():
        tabela.to_excel(writer, sheet_name=aba, index=False)
for aba, tabela in tabelas.items():
    nome = re.sub(r'[^a-z0-9]+', '_', aba.casefold()).strip('_')
    tabela.to_csv(SAIDA / f'{nome}.csv', index=False, encoding='utf-8-sig')
auditoria_linguistica.to_csv(SAIDA / 'auditoria_tokens.csv', index=False, encoding='utf-8-sig')
colunas_controle = [c for c in df.columns if not c.startswith('Tokens')]
df[colunas_controle].to_csv(SAIDA / 'observacoes_validadas.csv', index=False, encoding='utf-8-sig')

manifesto = {
    'executado_em_utc': datetime.now(timezone.utc).isoformat(),
    'arquivo_entrada': arquivo.name, 'sha256_entrada': hash_entrada,
    'n_respostas': len(df), 'n_cenarios_por_modelo': 10,
    'modelos_observados': configuracoes.to_dict('records'),
    'instrucao_comum_literal': instrucao,
    'acesso': 'interfaces web; sem novas requisições nesta análise',
    'idioma': 'português', 'temperatura': 'não controlada/documentada pela interface',
    'top_p': 'não controlado/documentado pela interface',
    'limite_tokens': 'não controlado/documentado pela interface',
    'sessoes': 'conversas novas declaradas no estudo; links privados ausentes e não reverificados neste pacote',
    'memoria_desativada': 'não comprovada por esta análise; não presumir',
    'json_locais_conferidos': len(registros),
    'diferencas_espacos_perifericos_excel_json': 'não verificadas neste pacote CSV',
    'inicio_coleta_utc': df['Inicio_UTC'].dropna().min() if registros else None,
    'fim_captura_utc': df['Captura_UTC'].dropna().max() if registros else None,
    'fora_80_120': int((~df['Dentro_80_120']).sum()),
    'novas_tentativas_declaradas': int(df['Nova_tentativa_declarada'].sum()),
    'protocolo_novas_tentativas': registro_tentativas.to_dict('records'),
    'metrica_principal': METRICA, 'alpha_principal': ALPHA,
    'suporte_principal': 'união lexical de cada dupla de respostas',
    'suportes_sensibilidade': ['par', 'comum por 20 respostas modelo/cenário'],
    'alphas': ALPHAS, 'logaritmo': 'natural', 'unidade': 'nat',
    'semente': SEMENTE, 'n_bootstrap': N_BOOTSTRAP, 'n_subamostras': N_SUBAMOSTRAS,
    'ic': 'percentil por cenários; marginal, não simultâneo',
    'wilcoxon': 'enumeração exata dos sinais dos postos; arredondamento numérico a 12 casas',
    'friedman': 'permutação exata dentro de cenário, programação dinâmica',
    'familias_holm': ['3 Wilcoxon principais', '3 sinais complementares',
                      '3 contrastes entre modelos', 'grade alpha/suporte', 'grade robustez'],
    'validacoes_executadas': True,
    'python': platform.python_version(),
    'versoes': {p: metadata.version(p) for p in ['numpy', 'pandas', 'spacy', 'scipy',
                                               'statsmodels', 'matplotlib', 'openpyxl']},
    'pipeline_pt': nlp.meta['version']
}
(SAIDA / 'manifesto_execucao.json').write_text(
    json.dumps(manifesto, ensure_ascii=False, indent=2), encoding='utf-8'
)
print('Resultados salvos em:', SAIDA.name)
if NO_COLAB:
    from google.colab import files
    files.download(str(SAIDA / 'analise_intra_inter_estatistica.xlsx'))


In [ ]:
# ============================================================
# 19. INTERPRETAÇÃO, LIMITAÇÕES E PENDÊNCIAS DO ARTIGO
# ============================================================

# O notebook produz evidências; não reescreve automaticamente conclusões do artigo.
# Um p não significativo não prova equivalência. Um p significativo não prova viés.
print('RESULTADOS PRINCIPAIS EFETIVAMENTE CALCULADOS')
display(testes_wilcoxon[['Modelo', 'Media_Diferenca', 'IC95_Inferior',
                        'IC95_Superior', 'p_Holm_Wilcoxon', 'p_Holm_Sinais']].round(6))
display(friedman_resultado.round(6))
print('LIMITAÇÕES A DECLARAR')
limitacoes = [
    'Dez cenários selecionados; generalização para outros prompts não demonstrada.',
    'Troca de nomes combina referência de gênero e identidade/associações nominais.',
    'Recorte binário; divergência lexical não constitui medida direta de viés.',
    'Web: versões internas parcialmente ocultas, parâmetros não controlados e cotas.',
    'Sessões novas declaradas não garantem independência estatística; os links não são redistribuídos.',
    'Comprimento, suporte lexical, suavização e lematização influenciam a medida.',
    'Filtros de sensibilidade podem selecionar respostas; análise principal mantém 600.',
    'Categorias confundem comprimento e conteúdo; apenas dois cenários longos.',
    'IC entre cenários não isolam toda a incerteza das dez gerações.',
    'Análise revisada após exploração anterior: não é protocolo pré-registrado.'
]
for item in limitacoes:
    print('-', item)
print('PENDÊNCIAS EXTERNAS AO NOTEBOOK')
print('Atualizar métodos, modelos observados, comando, datas e N=600 no manuscrito.')
print('Revisar tabelas/figuras, literatura acadêmica e referências; publicar materiais com autorização.')
print('Não transportar os modelos/comando do estudo histórico de 60 respostas para este corpus.')
print('Fontes metodológicas verificáveis:')
# Referências de base; a adequação dos testes ao delineamento é discutida acima.
# Demšar trata de classificadores/datasets: a aplicação aqui é uma adaptação por blocos,
# não uma validação específica de experimentos de gênero com LLMs.
print('DEMŠAR, Janez. Statistical Comparisons of Classifiers over Multiple Data Sets. '
      'Journal of Machine Learning Research, v. 7, p. 1–30, 2006.')
print('https://www.jmlr.org/papers/v7/demsar06a.html')
for url in [
    'https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.wilcoxon.html',
    'https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.friedmanchisquare.html',
    'https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.permutation_test.html',
    'https://spacy.io/usage/linguistic-features'
]:
    print(url)
